In [17]:
"""  --- Из репозитория авторов: исходный код IterGen --- """

#!git clone https://github.com/structuredllm/itergen.git

In [ ]:
"""  --- Из репозитория авторов: requirements.txt, ниже моя настройка установки зависимостей --- """

%pip install setuptools==80.9.0 wheel
%pip install --no-build-isolation -r itergen/requirements.txt

Note: you may need to restart the kernel to use updated packages.
  Cloning https://github.com/shubhamugare/mxeval.git to /tmp/pip-install-t7960nwq/mxeval_4d695b86598342d3adf1bcb315504d6d
  Running command git clone --filter=blob:none --quiet https://github.com/shubhamugare/mxeval.git /tmp/pip-install-t7960nwq/mxeval_4d695b86598342d3adf1bcb315504d6d
  Resolved https://github.com/shubhamugare/mxeval.git to commit 590129d063c5e7d145e8b15a83f00ff20d69a189
  Preparing metadata (setup.py) ... done
Note: you may need to restart the kernel to use updated packages.


In [ ]:
"""  --- Мой код: seed, пути к кэшу и подключение локальной библиотеки --- """

import sys, os

SEED = 228
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
from pathlib import Path

repo = Path("itergen").resolve()

cache_dir = repo.parent / "cache"
cache_dir.mkdir(parents=True, exist_ok=True)
os.environ["SYNCODE_CACHE"] = str(cache_dir) + os.sep
os.environ["HF_CACHE"] = str(cache_dir) + os.sep

for path in [
    repo,
    repo / "itergen" / "syncode",
    repo / "itergen" / "syncode" / "syncode",
]:
    sys.path.insert(0, str(path))
sys.modules.pop("itergen", None)

In [3]:
"""  --- Из IterGen и SynCode: Dataset, Grammar, Syncode и SyncodeLogitsProcessor --- """

import torch
import transformers
from itergen import Dataset
from itergen.syncode.syncode.infer import Syncode
from syncode.grammar_decoder import SyncodeLogitsProcessor
from syncode.parsers.grammars import Grammar

"""  --- Мои настройки: воспроизводимость и выбор точности модели --- """
transformers.set_seed(SEED)
torch.use_deterministic_algorithms(True)
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True
torch.backends.cuda.matmul.allow_tf32 = False

"""  --- Из библиотеки: готовая SQL-грамматика и загрузчик задач Spider --- """
sql_grammar = Grammar("sql")
sql_dataset = Dataset("spider", language="sql")
device = "cuda" if torch.cuda.is_available() else "cpu"

''' обычная генерация '''
"""  --- Из SynCode: загрузка модели; модель и параметры запуска выбираю я --- """
baseline = Syncode(
    model="Qwen/Qwen3.5-0.8B",
    mode="original",
    grammar="sql",
    parse_output_only=True,
    quantize=False,
    device=device,
    do_sample=False,
    max_new_tokens=200,
)
if device.startswith("cuda"):
    baseline.model.model.to(dtype=torch.float16)
baseline.model.model.eval()
print("Модель загружена. Примеров:", len(sql_dataset.problems))

''' Syncode генерация  '''
"""  --- Из SynCode: готовый процессор грамматических масок в режиме grammar_strict --- """
sql_decoder = SyncodeLogitsProcessor(
    grammar=sql_grammar,
    use_cache=True,
    tokenizer=baseline.model.tokenizer,
    parse_output_only=True,
    num_samples=1,
    mode="grammar_strict",
    dev_mode=True,
)
print("Грамматический процессор SynCode готов")

/home/nikita/proj/Test-task-RP/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d
Loading weights: 100%|██████████| 320/320 [00:00<00:00, 1756.34it/s]


Модель загружена. Примеров: 1034
Loading DFA mask store from /mnt/d/Test-task-RP-cache/mask_stores/TokenizersBackend/grammar_strict_6057483450_248044.pkl.
DFA mask store loaded from cache.
Грамматический процессор SynCode готов


In [4]:
"""  --- Мой код: общая функция генерации и обёртки для двух baseline --- """

MAX_NEW_TOKENS = 200
STOP_WORDS = ("\n\n",)

@torch.inference_mode()
def _generate(messages, max_new_tokens=MAX_NEW_TOKENS,
              stop_words=STOP_WORDS, decoder=None):
    tokenizer = baseline.model.tokenizer
    model = baseline.model.model

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False,
    )
    inputs = tokenizer(
        text,
        return_tensors="pt",
        add_special_tokens=False,
    ).to(next(model.parameters()).device)
    prompt_length = inputs["input_ids"].shape[1]

    processors = transformers.LogitsProcessorList()
    if decoder is not None:
        """  --- Из SynCode: сброс процессора и применение маски только к ответу --- """
        decoder.reset("")
        decoder.start_from = prompt_length
        processors.append(decoder)

    """  --- Из Transformers: генерация моделью с выбранными мной параметрами --- """
    output = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        num_beams=1,
        pad_token_id=model.generation_config.eos_token_id,
        repetition_penalty=1.0,
        logits_processor=processors,
        stop_strings=list(stop_words) if stop_words else None,
        tokenizer=tokenizer,
        return_dict_in_generate=False,
    )

    generated_ids = output[:, prompt_length:]
    raw_output = tokenizer.decode(generated_ids[0], skip_special_tokens=True)
    return raw_output, generated_ids.shape[1]


def generate_original(messages, max_new_tokens=MAX_NEW_TOKENS,
                      stop_words=STOP_WORDS):
    return _generate(
        messages, max_new_tokens=max_new_tokens, stop_words=stop_words,
        decoder=None,
    )


def generate_syncode(messages, max_new_tokens=MAX_NEW_TOKENS,
                     stop_words=STOP_WORDS):
    return _generate(
        messages, max_new_tokens=max_new_tokens, stop_words=stop_words,
        decoder=sql_decoder,
    )

In [5]:
"""  --- Мой код: выбор 50 основных и 5 отладочных задач, проверка пересечений и сохранение по seed --- """

import json
import random

RESULTS_DIR = Path("results") / f"seed_{SEED}"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
selection_path = RESULTS_DIR / "selection.json"

if selection_path.exists():
    selection = json.loads(selection_path.read_text(encoding="utf-8"))
    assert selection["seed"] == SEED
    eval_examples = selection["eval_examples"]
    debug_examples = selection["debug_examples"]
    selected_databases = selection["databases"]
else:
    r = random.Random(SEED)
    main_db = {}

    for source_id, problem in enumerate(sql_dataset.problems):
        db_id = problem["db_id"]
        example = dict(problem, source_id=source_id)

        if db_id not in main_db:
            main_db[db_id] = []
        
        main_db[db_id].append(example)

    extra_examples_db = []

    for db_id in main_db:
        examples = main_db[db_id]

        if len(examples) >= 11:
            extra_examples_db.append(db_id)

    extra_examples_db.sort()
    assert len(extra_examples_db) >= 5, "Недостаточно подходящих баз"

    ''' Рандомно выбираем 5 баз, и только те, где есть хотя бы 11 примеров (10 на базовые примеры и 1 на отладку) '''
    selected_databases = r.sample(extra_examples_db, 5)

    eval_examples = []
    debug_examples = []
    for db_id in selected_databases:
        examples = main_db[db_id]
        selected = r.sample(examples, 11)
        for i in range(10):
            eval_examples.append(selected[i])
    
        debug_examples.append(selected[10])

    r.shuffle(eval_examples)
    r.shuffle(debug_examples)
    assert len(eval_examples) == 50
    assert len(debug_examples) == 5

    # проверка на то, что у нас отладка не попала в основу
    eval_ids = []

    for example in eval_examples:
        eval_ids.append(example["source_id"])
    for example in debug_examples:
        assert example["source_id"] not in eval_ids

    print("Основная выборка:", len(eval_examples))
    print("Отладочная выборка:", len(debug_examples))
    print("Распределение по базам:")

    counts = {}
    for db_id in selected_databases:
        counts[db_id] = 0
    for example in eval_examples:
        db_id = example["db_id"]
        counts[db_id] += 1
    for db_id in counts:
        print(db_id, "-", counts[db_id])

    
    selection = {"seed": SEED, "databases": selected_databases,
                 "eval_examples": eval_examples, "debug_examples": debug_examples}
    selection_path.write_text(json.dumps(selection, ensure_ascii=False, indent=2) + "\n",
                              encoding="utf-8")

assert len(eval_examples) == 50 and len(debug_examples) == 5
assert not ({x["source_id"] for x in eval_examples} & {x["source_id"] for x in debug_examples})
print("Зафиксированная выборка:", SEED, len(eval_examples), "+", len(debug_examples))


Зафиксированная выборка: 228 50 + 5


In [6]:
"""  --- Мой код: подготовка SQLite-баз и выполнение SQL с ограничением времени --- """

import sqlite3
import time
import urllib.request
import zipfile
from pathlib import Path

DB_ROOT = cache_dir / "spider_databases"
SPIDER_ARCHIVE = cache_dir / "spider_data.zip"
SPIDER_URL = " "
selected_db_ids = set()
for example in eval_examples:
    selected_db_ids.add(example["db_id"])
for example in debug_examples:
    selected_db_ids.add(example["db_id"])

# Дополнительная база для отдельного разбора ошибки, вне eval/debug-метрик.
selected_db_ids.add("singer")

missing_db_ids = []
for db_id in sorted(selected_db_ids):
    db_file = DB_ROOT / db_id / f"{db_id}.sqlite"
    if not db_file.is_file():
        missing_db_ids.append(db_id)

if missing_db_ids:
    if not SPIDER_ARCHIVE.is_file():
        urllib.request.urlretrieve(SPIDER_URL, SPIDER_ARCHIVE)

    with zipfile.ZipFile(SPIDER_ARCHIVE) as archive:
        for db_id in missing_db_ids:
            target = DB_ROOT / db_id / f"{db_id}.sqlite"
            target.parent.mkdir(parents=True, exist_ok=True)
            name = f"spider_data/database/{db_id}/{db_id}.sqlite"
            target.write_bytes(archive.read(name))

def execute_sql(example, query):
    db_id = example["db_id"]
    db_path = DB_ROOT / db_id / f"{db_id}.sqlite"
    if not db_path.is_file():
        raise FileNotFoundError(db_path)

    database_uri = db_path.as_uri()
    database_uri += "?mode=ro"

    deadline = time.monotonic() + 5

    def check_time():
        if time.monotonic() > deadline:
            return 1
        return 0

    def decode_text(raw_text):
        return raw_text.decode("utf-8", errors="surrogateescape")

    connection = sqlite3.connect(database_uri, uri=True)
    connection.text_factory = decode_text
    try:
        connection.set_progress_handler(check_time, 10000)
        cursor = connection.execute(query)
        return cursor.fetchall()
    finally:
        connection.close()


## Baseline (обычная генерация и SynCode)

Ниже используются `Dataset` и `post_process_answer` из кода авторов. Цикл прогона, сохранение результатов и проверки я написал для этого задания отдельно от статьи. Оба режима получают один и тот же промпт, модель и лимит токенов. Сначала запускаются 5 debug- примеров, затем 50 eval.

`text_match` сравнивает SQL как текст после условной нормализации. `executable` означает, что SQLite выполнил запрос без ошибки, но это не показатель правильности ответа. Ниже отдельно считается execution accuracy на заполненных базах Spider, скаченных с [huggingface](https://huggingface.co/datasets/HAL-9001/spider-databases).<br>
Результаты выполнения сравниваются напрямую, с учётом порядка только для эталона с `ORDER BY`.


In [7]:
"""  --- Мой код: промпт, запуск baseline, замер времени, сохранение ответов и конфига --- """

import json
import re
from time import perf_counter
import hashlib
import platform 

RESULTS_DIR = Path("results") / f"seed_{SEED}"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
MODEL_ID = "Qwen/Qwen3.5-0.8B"
GENERATORS = {"original": generate_original, "syncode": generate_syncode}


def baseline_messages(example):
    return [
        {"role": "system", "content": "Return exactly one SQLite SELECT query. No Markdown or explanation."},
        {"role": "user", "content": example["prompt"]},
    ]


def normalized_sql(sql):
    return re.sub(r"\s+", " ", sql.strip().rstrip(";").strip()).casefold()


def run_baseline(examples, mode, split):
    path = RESULTS_DIR / f"{split}_{mode}.jsonl"
    by_id = {}
    if path.exists():
        for line in path.read_text(encoding="utf-8").splitlines():
            row = json.loads(line)
            assert row["source_id"] not in by_id, f"Повтор ID в {path}"
            by_id[row["source_id"]] = row

    assert set(by_id).issubset({x["source_id"] for x in examples}), "В файле другая выборка"
    with path.open("a", encoding="utf-8") as output:
        for example in examples:
            source_id = example["source_id"]
            messages = baseline_messages(example)
            if source_id in by_id:
                row = by_id[source_id]
                assert (row["prompt"] == example["prompt"] and row.get("input_messages") == messages
                        and row["mode"] == mode
                        and row["model_id"] == MODEL_ID and row["max_new_tokens"] == MAX_NEW_TOKENS
                        and row.get("stop_words") == list(STOP_WORDS)),f"Настройки или выборка изменились: {path}, ID {source_id}"
                continue

            if device.startswith("cuda"):
                torch.cuda.synchronize()
                
            started = perf_counter()
            raw_output, tokens = GENERATORS[mode](messages)
            if device.startswith("cuda"):
                torch.cuda.synchronize()
            seconds = perf_counter() - started
            
            """  --- Из авторского Dataset: обработка ответа модели перед моей проверкой SQL --- """
            sql = sql_dataset.post_process_answer(raw_output).strip()
            try:
                execute_sql(example, sql)
                executable, sql_error = True, None
            except (sqlite3.Error, FileNotFoundError) as error:
                executable, sql_error = False, str(error)

            row = {
                "source_id": source_id,
                "db_id": example["db_id"],
                "question": example["question"],
                "prompt": example["prompt"],
                "input_messages": messages,
                "gold_sql": example["ground_truth"],
                "mode": mode,
                "model_id": MODEL_ID,
                "max_new_tokens": MAX_NEW_TOKENS,
                "stop_words": list(STOP_WORDS),
                "raw_output": raw_output,
                "sql": sql,
                "tokens": tokens,
                "generated_tokens": tokens,
                "backtracks": 0,
                "seed": SEED,
                "seconds": seconds,
                "text_match": normalized_sql(sql) == normalized_sql(example["ground_truth"]),
                "executable": executable,
                "sql_error": sql_error,
            }
            output.write(json.dumps(row, ensure_ascii=False) + "\n")
            output.flush()
            by_id[source_id] = row

    return [by_id[example["source_id"]] for example in examples]


def print_summary(rows, mode):
    n = len(rows)
    print(f"{mode}: {n} примеров")
    print(f"  text_match: {sum(row['text_match'] for row in rows)}/{n}")
    print(f"  executable: {sum(row['executable'] for row in rows)}/{n}")
    print(f"  среднее число токенов: {sum(row['tokens'] for row in rows) / n:.1f}")
    print(f"  суммарное время: {sum(row['seconds'] for row in rows):.1f} с")


""" --- Читаем файл ноутбука --- """
notebook_source = json.loads(Path("Test_task_RP.ipynb").read_text(encoding="utf-8"))
source_text = "".join("".join(cell.get("source", [])) for cell in notebook_source["cells"])

""" --- Тут смотрим наш конфиг --- """
run_config = {
    "seed": SEED, 
    "model_id": MODEL_ID,
    "model_revision": getattr(baseline.model.model.config, "_commit_hash", None),
    "dtype": str(next(baseline.model.model.parameters()).dtype),
    "do_sample": False, 
    "max_new_tokens": MAX_NEW_TOKENS,
    "stop_words": list(STOP_WORDS), 
    "deterministic_algorithms": True,
    "torch": str(torch.__version__), 
    "transformers": transformers.__version__,
    "python": platform.python_version(),
    "device": torch.cuda.get_device_name(0) if device.startswith("cuda") else "cpu",
}
config_path = RESULTS_DIR / "run_config.json"
if config_path.exists():
    saved_config = json.loads(config_path.read_text(encoding="utf-8"))
    assert saved_config == run_config, "Код или среда изменились. Используйте отдельную папку прогона."
else:
    config_path.write_text(json.dumps(run_config, ensure_ascii=False, indent=2) + "\n",
                           encoding="utf-8")


In [8]:
"""  --- Мой код: запуск двух baseline на отладочных задачах и вывод ответов --- """

debug_results = {
    mode: run_baseline(debug_examples, mode, "debug") for mode in GENERATORS
}

for index, example in enumerate(debug_examples):
    print("\n" + "-" * 70)
    print("ID:", example["source_id"], "База:", example["db_id"])
    print("Вопрос:", example["question"])
    print("Эталонный SQL:", example["ground_truth"])
    for mode in GENERATORS:
        row = debug_results[mode][index]
        print(f"\n{mode} — полный ответ модели:\n{row['raw_output']}")
        print("SQL после обработки:", row["sql"])
        print("text_match:", row["text_match"], "executable:", row["executable"])
        print("Ошибка SQL:", row["sql_error"])
        print("Токенов:", row["tokens"], "Время, с:", round(row["seconds"], 2))



----------------------------------------------------------------------
ID: 754 База: world_1
Вопрос: What are the countries where either English or Dutch is the official language ?
Эталонный SQL: select country.name from country join countrylanguage on country.code = countrylanguage.countrycode where countrylanguage.language = 'english' and isofficial = 't' union select country.name from country join countrylanguage on country.code = countrylanguage.countrycode where countrylanguage.language = 'dutch' and isofficial = 't'

original — полный ответ модели:
SELECT c.code, c.name, c.continent, c.region, c.surfacearea, c.indepyear, c.population, c.lifeexpectancy, c.gnp, c.gnpold, c.localname, c.governmentform, c.headofstate, c.capital, c.code2
FROM country c
JOIN countrylanguage cl ON c.code = cl.countrycode
WHERE cl.isofficial = '1' OR cl.isofficial = '0';

SQL после обработки: SELECT c.code, c.name, c.continent, c.region, c.surfacearea, c.indepyear, c.population, c.lifeexpectancy, c.gnp,

In [9]:
"""  --- Мой код: запуск на 50 задачах и сохранение сводных метрик --- """

eval_results = {
    mode: run_baseline(eval_examples, mode, "eval")
    for mode in GENERATORS
}

for mode, rows in eval_results.items():
    print_summary(rows, mode)

summary = {
    "model_id": MODEL_ID,
    "selection_seed": SEED,
    "eval_examples": len(eval_examples),
    "databases": selected_databases,
    "metrics": {
        mode: {
            "text_match": sum(row["text_match"] for row in rows),
            "executable": sum(row["executable"] for row in rows),
            "total": len(rows),
            "average_tokens": sum(row["tokens"] for row in rows) / len(rows),
            "total_seconds": sum(row["seconds"] for row in rows),
        }
        for mode, rows in eval_results.items()
    }
}
(RESULTS_DIR / "summary.json").write_text(
    json.dumps(summary, ensure_ascii=False, indent=2) + "\n", encoding="utf-8"
)


original: 50 примеров
  text_match: 1/50
  executable: 28/50
  среднее число токенов: 50.9
  суммарное время: 148.2 с
syncode: 50 примеров
  text_match: 1/50
  executable: 12/50
  среднее число токенов: 90.7
  суммарное время: 239.0 с


549

In [ ]:
"""  --- Мой код: сравнение результатов SQL и оценка выборок, оценщик Spider авторов здесь не используется --- """

import sqlite3
import json
import re
from collections import Counter

def check_execution(example: dict, predicted_sql: str) -> dict:
    """Сравнивает результаты эталонного и предсказанного SQL-запросов."""
    gold_sql = example["ground_truth"]
    
    try:
        gold_rows = execute_sql(example, gold_sql)
    except sqlite3.Error as e:
        raise RuntimeError(f"Эталонный SQL не выполнился: {e}")

    try:
        predicted_rows = execute_sql(example, predicted_sql)
    except sqlite3.Error as error:
        return {
            "execution_match": False, 
            "executable": False, 
            "sql_error": str(error)
        }

    # Проверяем, есть ли сортировка в эталонном запросе
    has_order_by = bool(re.search(r"\border\s+by\b", gold_sql, re.IGNORECASE))
    
    if has_order_by:
        is_match = predicted_rows == gold_rows
    else:
        is_match = Counter(predicted_rows) == Counter(gold_rows)

    return {
        "execution_match": is_match,
        "executable": True,
        "sql_error": None,
        "gold_row_count": len(gold_rows),
        "predicted_row_count": len(predicted_rows),
    }

def evaluate_split(examples: list, results: dict, split_name: str) -> dict:
    """Оценивает результаты для конкретного прогона (eval/debug) по всем моделям"""
    scored_by_mode = {}
    
    for mode, rows in results.items():
        assert len(examples) == len(rows), "Количество примеров и результатов не совпадает"
        
        scored_rows = []
        for example, row in zip(examples, rows):
            assert example["source_id"] == row["source_id"], "Несовпадение ID источника"
            
            score = check_execution(example, row["sql"])
            scored_rows.append({
                "source_id": row["source_id"],
                "db_id": row["db_id"],
                "mode": mode,
                **score
            })
            
        # Сохраняем детальные результаты в JSONL
        file_path = RESULTS_DIR / f"{split_name}_execution_{mode}.jsonl"
        with file_path.open("w", encoding="utf-8") as f:
            for item in scored_rows:
                f.write(json.dumps(item, ensure_ascii=False) + "\n")
                
        scored_by_mode[mode] = scored_rows
        
    return scored_by_mode



""" --- Основная логика выполнения --- """

eval_execution = evaluate_split(eval_examples, eval_results, "eval")
debug_execution = evaluate_split(debug_examples, debug_results, "debug")

for mode, rows in eval_execution.items():
    correct = sum(r["execution_match"] for r in rows)
    print(f"{mode} execution accuracy: {correct} / {len(rows)}")

for mode, rows in debug_execution.items():
    debug_info = [(r["source_id"], r["execution_match"]) for r in rows]
    print(f"{mode} debug execution: {debug_info}")

""" --- Обновление и сохранение summary --- """

summary["database_source"] = SPIDER_URL

for mode, rows in eval_execution.items():
    total = len(rows)
    correct = sum(r["execution_match"] for r in rows)
    executable = sum(r["executable"] for r in rows)
    
    summary["metrics"][mode]["execution_correct"] = correct
    summary["metrics"][mode]["execution_accuracy"] = correct / total if total > 0 else 0.0
    summary["metrics"][mode]["executable"] = executable

summary_path = RESULTS_DIR / "summary.json"
with summary_path.open("w", encoding="utf-8") as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)
    f.write("\n")

original execution accuracy: 14 / 50
syncode execution accuracy: 7 / 50
original debug execution: [(754, True), (580, False), (230, False), (863, True), (456, False)]
syncode debug execution: [(754, False), (580, False), (230, False), (863, True), (456, False)]


## Мой цикл IterGen
Авторская библиотека IterGen используется как база `start / forward / view / backward`, но для совместимость с Transformers 5.3 и кэшем Qwen3.5 я ее пропатчил (об этом было выше). После отката Qwen пересчитывает состояние модели по сохранённому префиксу.

Проверяем завершённые имена таблиц и столбцов по схеме текущей базы. Сначала до двух раз откатываем только ошибочное имя. Если в том же месте ошибка повторяется 3 раз, то откатываемся квыражению после `SELECT` => наше ограничение - наличие `SELECT` в запросе. Общие ограничения шагов, откатов и токенов исключают бесконечные повторения.


In [ ]:
"""  --- Из библиотеки: класс IterGen с нашими патчами совместимости; сам контроллер ниже написан отдельно --- """

from itergen.main import IterGen

"""  --- Мои ограничения: число шагов, откатов, токенов и локальных попыток --- """
MAX_ITER_STEPS = 25
MAX_BACKTRACKS = 8
MAX_TOTAL_ITER_TOKENS = 300
LOCAL_RETRIES = 2

"""  --- Функция поиска всех столбцов и таблиц --- """
"""  --- Мой код: схема из SQLite; в SQL-примере авторов схема разбирается из db_info --- """
def schema_names(example):
    db_id = example["db_id"]
    db_path = DB_ROOT / db_id / f"{db_id}.sqlite"
    connection = sqlite3.connect(db_path.as_uri() + "?mode=ro", uri=True)
    try:
        names = [name for (name,) in connection.execute(
            "SELECT name FROM sqlite_master WHERE type = 'table' AND name NOT LIKE 'sqlite_%'"
        )]
        tables = {name.casefold() for name in names}
        columns = set()
        for name in names:
            quoted = name.replace('"', '""')
            columns.update(row[1].casefold() for row in connection.execute(
                f'PRAGMA table_info("{quoted}")'
            ))
        return tables, columns
    finally:
        connection.close()


"""  --- Моя проверка имён по схеме, по идее SQL-примера авторов; без разрешения алиасов --- """
def valid_name(unit, value, tables, columns):
    name = value.strip().strip('"`').casefold()
    if unit == "table_name":
        return name in tables
    column = name.rsplit(".", 1)[-1].strip('"`')
    return column == "*" or column in columns


"""  --- Мой контроллер: выбор проверок, глубины отката, бюджетов и запись трассы --- """
def generate_itergen(example, generator, max_backtracks=MAX_BACKTRACKS, deep_rollback=True):
    if device.startswith("cuda"):
        torch.cuda.synchronize()
    started = perf_counter()
    tables, columns = schema_names(example)
    """  --- Из IterGen: start начинает сессию генерации --- """
    generator.start(baseline_messages(example))
    
    trace = []
    local_attempts = {}
    backtracks = 0
    status = "step_limit"

    for step in range(1, MAX_ITER_STEPS + 1):
        if generator.finished():
            retained = generator.session_tokens.shape[-1] - generator.prompt_tokens.shape[-1]
            status = "max_output_tokens" if retained >= MAX_NEW_TOKENS else "stopped"
            break
        if generator._metadata["total_tokens"] >= MAX_TOTAL_ITER_TOKENS:
            status = "token_budget"
            break
        
        """  --- Из IterGen: view читает завершённые имена, forward дописывает до следующего имени --- """
        before = {unit: len(generator.view(unit)[0])
                  for unit in ("table_name", "column_name")}
        output = generator.forward(units=["table_name", "column_name"], num=1)[0]
        rejected = False
        
        for unit in ("table_name", "column_name"):
            new_names = generator.view(unit)[0][before[unit]:]
            
            for value in new_names:
                valid = valid_name(unit, value, tables, columns)
                
                event = {"step": step, "unit": unit, "value": value,
                         "valid": valid, "before": output}
                if valid:
                    event["action"] = "continue"
                else:
                    """  --- Из IterGen: backward удаляет имя, решение об откате и счётчики мои --- """
                    generator.backward(unit)
                    backtracks += 1
                    
                    prefix = generator.structured_gen[0]
                    key = (unit, prefix)
                    local_attempts[key] = local_attempts.get(key, 0) + 1
                    event["action"] = f"backward_{unit}"
                    if deep_rollback and local_attempts[key] > LOCAL_RETRIES and generator.view("select_expr")[0] and backtracks < max_backtracks:
                        """  --- Моё правило глубокого отката; возврат по грамматике выполняет IterGen --- """
                        generator.backward("select_expr")
                        backtracks += 1
                        event["action"] += "_then_select_expr"
                    
                    event["after"] = generator.structured_gen[0]
                    rejected = True
                trace.append(event)
                if rejected:
                    break
                
            if rejected:
                break
            
        if backtracks >= max_backtracks:
            status = "backtrack_limit"
            break
    
        if not rejected and generator.finished():
            retained = generator.session_tokens.shape[-1] - generator.prompt_tokens.shape[-1]
            status = "max_output_tokens" if retained >= MAX_NEW_TOKENS else "stopped"
            break

    raw_output = generator.structured_gen[0]
    stop_reason = status
    if status in ("stopped", "max_output_tokens"):
        stop_reason = "grammar_complete" if generator.inc_parsers[0].symbol_pos_map.get_symbol_count("start") else "eos_token"
        for criterion in generator.stopping_criteria:
            if bool(criterion(generator.session_tokens, ())[0]):
                stop_reason = type(criterion).__name__
                break
    if device.startswith("cuda"):
        torch.cuda.synchronize()
    return {
        "raw_output": raw_output,
        "sql": sql_dataset.post_process_answer(raw_output).strip(),
        "trace": trace,
        "status": status,
        "stop_reason": stop_reason,
        "retained_tokens": generator.session_tokens.shape[-1] - generator.prompt_tokens.shape[-1],
        "backtracks": backtracks,
        "generated_tokens": generator._metadata["total_tokens"],
        "seconds": perf_counter() - started,
    }


"""  --- Мой код: создание библиотечного IterGen с общими весами, токенизатором и DFA --- """
def make_itergen():
    # Reuse the loaded model, tokenizer and DFA to avoid a second large copy.
    return IterGen(
        grammar="sql", model_id=MODEL_ID, device=device, quantize=False,
        model=baseline.model.model, tokenizer=baseline.model.tokenizer,
        dfa_mask_store=sql_decoder.dfa_mask_store,
        do_sample=False, num_beams=1, num_return_sequences=1,
        max_new_tokens=MAX_NEW_TOKENS, max_tokens=1024,
        stop_strings=list(STOP_WORDS), recurrence_penalty=0.3,
    )


In [12]:
"""  --- Мой код: параметры контроллера, продолжение прогона и сохранение отладочных трасс --- """

itergen_path = RESULTS_DIR / "debug_itergen.jsonl"
cycle_config = {
    "cycle_version": 1,
    "model_id": MODEL_ID, "max_new_tokens": MAX_NEW_TOKENS,
    "max_steps": MAX_ITER_STEPS, "max_backtracks": MAX_BACKTRACKS,
    "max_total_tokens": MAX_TOTAL_ITER_TOKENS, "local_retries": LOCAL_RETRIES,
    "recurrence_penalty": 0.3,
}

"""  --- Загрузка старых кэшей --- """
itergen_rows = {}
if itergen_path.exists():
    for line in itergen_path.read_text(encoding="utf-8").splitlines():
        row = json.loads(line)
        
        assert row["source_id"] not in itergen_rows
        itergen_rows[row["source_id"]] = row

"""  --- Загрузка задач, которые надо сделать --- """
pending = []
for example in debug_examples:
    row = itergen_rows.get(example["source_id"])
    if row is None:
        pending.append(example)
    else:
        assert row["prompt"] == example["prompt"] and row["controller_config"] == cycle_config


"""  --- Init itergen --- """
if pending:
    iter_gen = make_itergen()

    with itergen_path.open("a", encoding="utf-8") as output:
        for example in pending:
            source_id = example["source_id"]
            result = generate_itergen(example, iter_gen)
            assert result["generated_tokens"] > 0, (
                f"ID {source_id}: генерация завершилась до первого токена. "
                "Перезапустите ядро, чтобы загрузить исправленный IterGen."
            )
            row = {
                "source_id": source_id, 
                "db_id": example["db_id"],
                "question": example["question"], 
                "prompt": example["prompt"],
                "gold_sql": example["ground_truth"], 
                "controller_config": cycle_config,
                "seed": SEED,
                **result, 
                **check_execution(example, result["sql"])
            }
            
            output.write(json.dumps(row, ensure_ascii=False) + "\n")
            output.flush()
            
            itergen_rows[source_id] = row
            print(source_id, row["status"], "backtracks", row["backtracks"],
                      "tokens", row["generated_tokens"], "execution_match", row["execution_match"])

print("Отладочных примеров IterGen:", len(itergen_rows))

754 stopped backtracks 0 tokens 94 execution_match True
580 stopped backtracks 0 tokens 55 execution_match False
230 stopped backtracks 0 tokens 66 execution_match False
863 stopped backtracks 0 tokens 11 execution_match True
456 stopped backtracks 2 tokens 74 execution_match False
Отладочных примеров IterGen: 5


In [13]:
"""  --- Мой код: проверки лимитов, сводка и вывод ошибок IterGen на отладке --- """

ordered_itergen_rows = [itergen_rows[example["source_id"]] for example in debug_examples]
assert len(ordered_itergen_rows) == 5
for row in ordered_itergen_rows:
    assert 0 < row["generated_tokens"] <= MAX_TOTAL_ITER_TOKENS + MAX_NEW_TOKENS, (
        f"ID {row['source_id']}: число токенов {row['generated_tokens']}, статус {row['status']}"
    )
    assert row["backtracks"] <= MAX_BACKTRACKS, (
        f"ID {row['source_id']}: откатов {row['backtracks']}, лимит {MAX_BACKTRACKS}"
    )
assert all(check_execution(example, example["ground_truth"])["execution_match"]
           for example in debug_examples)

itergen_debug_summary = {
    "examples": len(ordered_itergen_rows),
    "execution_correct": sum(row["execution_match"] for row in ordered_itergen_rows),
    "execution_accuracy": sum(row["execution_match"] for row in ordered_itergen_rows) / len(ordered_itergen_rows),
    "total_backtracks": sum(row["backtracks"] for row in ordered_itergen_rows),
    "total_generated_tokens": sum(row["generated_tokens"] for row in ordered_itergen_rows),
    "statuses": {str(row["source_id"]): row["status"] for row in ordered_itergen_rows},
}
(RESULTS_DIR / "itergen_debug_summary.json").write_text(
    json.dumps(itergen_debug_summary, ensure_ascii=False, indent=2) + "\n",
    encoding="utf-8",
)
print("IterGen debug:", itergen_debug_summary)
for row in ordered_itergen_rows:
    print("\nID", row["source_id"], "SQL:", row["sql"])
    for event in row["trace"]:
        if not event["valid"]:
            print("  step", event["step"], event["unit"], repr(event["value"]),
                  "->", event["action"])


IterGen debug: {'examples': 5, 'execution_correct': 2, 'execution_accuracy': 0.4, 'total_backtracks': 2, 'total_generated_tokens': 300, 'statuses': {'754': 'stopped', '580': 'stopped', '230': 'stopped', '863': 'stopped', '456': 'stopped'}}

ID 754 SQL: SELECT c.code, c.name, c.continent, c.region, c.surfacearea, c.indepyear, c.population, c.lifeexpectancy, c.gnp, c.gnpold, c.localname, c.governmentform, c.headofstate, c.capital, c.code2 FROM country c JOIN countrylanguage cl ON c.code = cl.countrycode WHERE cl.isofficial =1 OR cl.isofficial =0;

ID 580 SQL: SELECT DISTINCT s.first_name, s.middle_name, s.last_name, s.cell_mobile_number, s.email_address, s.ssn, s.date_first_registered, s.date_left, s.other_student_details FROM students ORDER BY s.other_student_details ASC;

ID 230 SQL: SELECT a.airline, COUNT(f.flightno) as flight_count FROM airlines a JOIN airports a2 ON a.airline = a2.airline JOIN flights f ON a2.airportcode = f.sourceairport GROUP BY a.airline ORDER BY flight_count DE

## Сравнение и абляции

Я заранее выбрал три варианта: основной цикл с 8 откатами, тот же цикл с 2 откатами и цикл с 8 откатами без углубления до `SELECT`. Модель, промпт, 50 примеров, предел 200 токенов ответа и остальные настройки одинаковы.

Проверки таблиц и столбцов происходят после каждого завершённого имени. Две ошибки в одном месте, как и до этого, определяют переход к более глубокому откату. Лимит шагов это число вызовов `forward`, а не время в секундах. Общий бюджет токенов проверяется между вызовами `forward`, поэтому возможен ограниченный перерасход внутри последнего вызова.

`stopped` означает, что сработал механизм завершения, он не оценивает правильность SQL. Новое поле `stop_reason` хранит конкретную причину, например `EosTokenCriteria` или `StopStringCriteria`. Качество оценивает только отдельная проверка выполнения.


In [14]:
"""  --- Мой код: повторные запуски для проверки совпадения ответов, токенов и трасс --- """

# Проверяем повторяемость на одном debug-примере, не подбирая по нему параметры
repro_example = debug_examples[0]
repeatability = {}
for mode, generate in GENERATORS.items():
    first, first_tokens = generate(baseline_messages(repro_example))
    second, second_tokens = generate(baseline_messages(repro_example))
    repeatability[mode] = {"same_answer": first == second,
                           "same_token_count": first_tokens == second_tokens}
    assert first == second and first_tokens == second_tokens, f"Неповторяемый результат {mode}"

iter_gen = make_itergen()
first = generate_itergen(repro_example, iter_gen)
second = generate_itergen(repro_example, iter_gen)
repeatability["itergen"] = {"same_answer": first["raw_output"] == second["raw_output"],
                           "same_token_count": first["generated_tokens"] == second["generated_tokens"],
                           "same_trace": first["trace"] == second["trace"]}

assert all(repeatability["itergen"].values()), "Неповторяемый результат IterGen"
(RESULTS_DIR / "repeatability.json").write_text(
    json.dumps(repeatability, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
print("Повторяемость:", repeatability)


Повторяемость: {'original': {'same_answer': True, 'same_token_count': True}, 'syncode': {'same_answer': True, 'same_token_count': True}, 'itergen': {'same_answer': True, 'same_token_count': True, 'same_trace': True}}


In [ ]:
"""  --- Мой код: варианты абляций и общий запуск IterGen с сохранением результатов --- """

VARIANTS = {
    "itergen": {"max_backtracks": 8, "deep_rollback": True},
    "itergen_2_backtracks": {"max_backtracks": 2, "deep_rollback": True},
    "itergen_local_only": {"max_backtracks": 8, "deep_rollback": False},
}


def run_itergen(examples, split, name, settings, generator):
    path = RESULTS_DIR / f"{split}_{name}.jsonl"
    config = {**cycle_config, **settings}
    by_id = {}
    
    # Чтобы еще раз не проходить по одному и тому же примеру
    if path.exists():
        for line in path.read_text(encoding="utf-8").splitlines():
            row = json.loads(line)
            assert row["source_id"] not in by_id
            by_id[row["source_id"]] = row
    assert set(by_id).issubset({x["source_id"] for x in examples})
    
    with path.open("a", encoding="utf-8") as output:
        for example in examples:
            source_id = example["source_id"]
            
            # Если ответ уже есть, проверяем cfg и промт
            if source_id in by_id:
                row = by_id[source_id]
                assert row["controller_config"] == config and row["prompt"] == example["prompt"]
                continue
            
            result = generate_itergen(example, generator, **settings)
            assert result["generated_tokens"] > 0, f"Нулевой ответ, ID {source_id}"
            assert result["backtracks"] <= settings["max_backtracks"]
            
            row = {"source_id": source_id, "db_id": example["db_id"],
                   "question": example["question"], "prompt": example["prompt"],
                   "gold_sql": example["ground_truth"], "seed": SEED,
                   "controller_config": config, **result,
                   **check_execution(example, result["sql"])}
            
            output.write(json.dumps(row, ensure_ascii=False) + "\n")
            output.flush()
            by_id[source_id] = row
            
    return [by_id[x["source_id"]] for x in examples]


comparison_results = dict(eval_results)
for name, settings in VARIANTS.items():
    rows = run_itergen(eval_examples, "eval", name, settings, iter_gen)
    comparison_results[name] = rows
    print(name, "завершён:", sum(r["execution_match"] for r in rows), "/", len(rows))


itergen завершён: 14 / 50
itergen_2_backtracks завершён: 13 / 50


In [ ]:
"""  --- Мой код: итоговые метрики, попарное сравнение абляций и запись JSON/CSV --- """

import csv


def summarize_experiment(rows):
    """  --- Список артифактов каждого режима --- """
    
    scores = [check_execution({"db_id": r["db_id"], "ground_truth": r["gold_sql"]}, r["sql"])
              for r in rows]
    
    correct = sum(score["execution_match"] for score in scores)
    executable = sum(score["executable"] for score in scores)
    n = len(rows)
    tokens = sum(r.get("generated_tokens", r.get("tokens", 0)) for r in rows)
    backtracks = sum(r.get("backtracks", 0) for r in rows)
    
    return {"examples": n, "executable": executable, "executable_rate": executable / n,
            "execution_correct": correct, "execution_accuracy": correct / n,
            "total_seconds": sum(r["seconds"] for r in rows),
            "average_seconds": sum(r["seconds"] for r in rows) / n,
            "total_generated_tokens": tokens, "average_tokens": tokens / n,
            "total_backtracks": backtracks, "average_backtracks": backtracks / n}


comparison = {name: summarize_experiment(rows) for name, rows in comparison_results.items()}

"""  --- Сохраняем в json и csv ( csv для удобного просмотра ) --- """
(RESULTS_DIR / "comparison.json").write_text(
    json.dumps({"seed": SEED, "metrics": comparison}, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")

with (RESULTS_DIR / "comparison.csv").open("w", encoding="utf-8", newline="") as output:
    writer = csv.DictWriter(output, fieldnames=["mode"] + list(next(iter(comparison.values())).keys()))
    writer.writeheader()
    for mode, metrics in comparison.items():
        writer.writerow({"mode": mode, **metrics})

"""  --- У одной и той же задачи сравниваем основной вариант с абляцией --- """
paired_ablation = {}
full = comparison_results["itergen"]

for name in ("itergen_2_backtracks", "itergen_local_only"):
    improved, worsened, unchanged = [], [], []
    
    for main, other in zip(full, comparison_results[name]):
        assert main["source_id"] == other["source_id"]
        
        target = improved if main["execution_match"] and not other["execution_match"] else (
            worsened if other["execution_match"] and not main["execution_match"] else unchanged)
        
        target.append(main["source_id"])
        
    paired_ablation[name] = {"full_better": improved, "full_worse": worsened, "same": unchanged}

(RESULTS_DIR / "ablation_pairs.json").write_text(
    json.dumps(paired_ablation, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")

for mode, metrics in comparison.items():
    print(mode, "executable", metrics["executable"], "/ 50; correct", metrics["execution_correct"],
          "/ 50; backtracks", metrics["total_backtracks"], "; tokens", metrics["total_generated_tokens"])


# Общий JSON для всех трёх режимов и абляций, включая число откатов
summary["metrics"] = comparison
summary["stop_reasons"] = {
    mode: dict(Counter(row.get("stop_reason", "baseline") for row in rows))
    for mode, rows in comparison_results.items()
}
summary["selection_file"] = str(selection_path)
summary["ablation_pairs_file"] = str(RESULTS_DIR / "ablation_pairs.json")

(RESULTS_DIR / "summary.json").write_text(
    json.dumps(summary, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")


original executable 28 / 50; correct 14 / 50; backtracks 0 ; tokens 2543
syncode executable 12 / 50; correct 7 / 50; backtracks 0 ; tokens 4535
itergen executable 28 / 50; correct 14 / 50; backtracks 18 ; tokens 2514
itergen_2_backtracks executable 27 / 50; correct 13 / 50; backtracks 16 ; tokens 2362
itergen_local_only executable 28 / 50; correct 14 / 50; backtracks 18 ; tokens 2514


In [ ]:
"""  --- Мой код: отдельный пример для глубоких откатов, который не входит в общую оценку --- """

# Дополнительный случай из Spider для изучения глубоких откатов, вне общей метрики.
extra_example = dict(sql_dataset.problems[1029], source_id=1029)

assert extra_example["source_id"] not in {x["source_id"] for x in eval_examples + debug_examples}

extra_itergen = run_itergen([extra_example], "extra", "itergen", VARIANTS["itergen"], iter_gen)
print("Дополнительный пример:", extra_itergen[0]["status"],
      "откатов", extra_itergen[0]["backtracks"])


Дополнительный пример: backtrack_limit откатов 8


## Что воспроизведено и что означает эксперимент

В [статье IterGen](https://arxiv.org/pdf/2410.07295) откат идёт по границе единицы грамматики, т.е. сохраняется префикс ответа, ошибочное продолжение удаляется. Повторная генерация всего SQL удаляла бы и этот префикс. Мой вариант выбирает момент проверки и точку отката "сам".

Внешняя проверка выполняется, когда имя полностью сгенерировано, ну это очевидно, что неполное слово ещё нельзя проверить на наличие в схеме. Проверка имён ловит ссылки на несуществующие объекты, но не гарантирует правильность фильтра, соединения или агрегата, то есть проверка больше синтаксическая, не семантическая. Семантику мы проверяем при выполнении целевого запроса и того, что нам выдана модель.

Использованы те же входные сообщения и веса модели, но SynCode всегда маскирует токены, а авторский IterGen применяет маску при нарушении грамматики. Для Qwen3.5 после отката пересчитывается состояние модели по сохранённому префиксу, поэтому скорость не воспроизводит оптимизацию KV-кэша статьи.
(Подробнее в REPORT.md)
